# Solution 10 — a guarded helpdesk

One agent, three rules: every tool call is logged into deps, deletes wait for a person, and a run
may make at most 3 tool calls. Exercise from
[`../notebooks/10_capabilities_and_guardrails.ipynb`](../notebooks/10_capabilities_and_guardrails.ipynb).
Runs keyless.

In [ ]:
from learningdantic import setup

MODEL = setup()

## The agent

`field(default_factory=list)` gives every new `Desk` its own empty list. The hook reads that list from
`ctx.deps`, so each run's log lands in the `Desk` you pass in.

In [ ]:
from dataclasses import dataclass, field

from pydantic_ai import Agent, DeferredToolRequests, RunContext
from pydantic_ai.capabilities import Hooks
from pydantic_ai.models.test import TestModel


@dataclass
class Desk:
    audit_log: list[str] = field(default_factory=list)


policy = Hooks()


@policy.on.before_tool_execute
def audit(ctx: RunContext[Desk], call, tool_def, args):
    ctx.deps.audit_log.append(call.tool_name)
    return args


helpdesk = Agent(
    TestModel(),
    deps_type=Desk,
    output_type=[str, DeferredToolRequests],
    capabilities=[policy],
)


@helpdesk.tool_plain
def read_ticket(ticket_id: int) -> str:
    """Read one support ticket."""
    return f"ticket {ticket_id}: the printer on floor 2 is jammed"


@helpdesk.tool_plain(requires_approval=True)
def delete_ticket(ticket_id: int) -> str:
    """Delete a ticket for good."""
    return f"ticket {ticket_id} deleted"

## Part 1 — audit + approval

`TestModel` calls both tools. `read_ticket` runs at once; `delete_ticket` waits for us.

In [ ]:
from pydantic_ai import ToolDenied

desk = Desk()
paused = await helpdesk.run("Delete ticket 2.", deps=desk)
assert isinstance(paused.output, DeferredToolRequests), paused.output
assert [call.tool_name for call in paused.output.approvals] == ["delete_ticket"]

reason = "A manager must approve deletes."
denied = paused.output.build_results(
    approvals={call.tool_call_id: ToolDenied(reason) for call in paused.output.approvals}
)
final = await helpdesk.run(
    message_history=paused.all_messages(), deferred_tool_results=denied, deps=desk
)
print("final output:", final.output)
print("audit log   :", desk.audit_log)

assert reason in final.output, final.output
assert "delete_ticket" not in desk.audit_log, desk.audit_log
assert desk.audit_log == ["read_ticket"], desk.audit_log

## Part 2 — tool-call cap

The stuck model asks for `read_ticket` forever. The cap lets 3 calls run and stops the 4th before it runs,
so the hook logs exactly 3.

In [ ]:
from pydantic_ai import UsageLimitExceeded, UsageLimits
from pydantic_ai.messages import ModelResponse, ToolCallPart
from pydantic_ai.models.function import FunctionModel


def stuck(messages, info) -> ModelResponse:
    """A fake model that never stops calling read_ticket."""
    return ModelResponse(parts=[ToolCallPart("read_ticket", {"ticket_id": 2})])


desk = Desk()
stopped = False
try:
    await helpdesk.run(
        "Summarise ticket 2.",
        deps=desk,
        model=FunctionModel(stuck),
        usage_limits=UsageLimits(tool_calls_limit=3),
    )
except UsageLimitExceeded as e:
    stopped = True
    print("stopped:", e)

assert stopped, "the tool-call cap did not stop the run"
assert desk.audit_log == ["read_ticket"] * 3, desk.audit_log

## Stretch — approve only allow-listed tickets

A scripted model asks to delete tickets 2 and 9 in one step, then reports the tool results it got back.
Only ticket 2 is on the allow-list.

In [ ]:
from pydantic_ai.messages import TextPart, ToolReturnPart

ALLOWED = {2}


def deleter(messages, info) -> ModelResponse:
    """First turn: ask to delete two tickets. After the results come back: report them."""
    results = [str(p.content) for p in messages[-1].parts if isinstance(p, ToolReturnPart)]
    if results:
        return ModelResponse(parts=[TextPart(" | ".join(results))])
    return ModelResponse(
        parts=[
            ToolCallPart("delete_ticket", {"ticket_id": 2}),
            ToolCallPart("delete_ticket", {"ticket_id": 9}),
        ]
    )


desk = Desk()
paused = await helpdesk.run("Delete tickets 2 and 9.", deps=desk, model=FunctionModel(deleter))
decisions = paused.output.build_results(
    approvals={
        call.tool_call_id: True
        if call.args_as_dict()["ticket_id"] in ALLOWED
        else ToolDenied("Not on the allow-list.")
        for call in paused.output.approvals
    }
)
final = await helpdesk.run(
    message_history=paused.all_messages(),
    deferred_tool_results=decisions,
    deps=desk,
    model=FunctionModel(deleter),
)
print("final output:", final.output)
print("audit log   :", desk.audit_log)

assert "ticket 2 deleted" in final.output, final.output
assert "Not on the allow-list." in final.output, final.output
assert "ticket 9 deleted" not in final.output, final.output
assert desk.audit_log == ["delete_ticket"], desk.audit_log